In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/jwang2025learning/lr-tuning/lr_best_predict.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/y_train.pkl
/kaggle/input/notebooks/jwang2025learning/lr-tuning/__results__.html
/kaggle/input/notebooks/jwang2025learning/lr-tuning/y_tr.pkl
/kaggle/input/notebooks/jwang2025learning/lr-tuning/submission.csv
/kaggle/input/notebooks/jwang2025learning/lr-tuning/lr_val_predict.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/__notebook__.ipynb
/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_test.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/__output__.json
/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_val.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_train.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_tr.npy
/kaggle/input/notebooks/jwang2025learning/lr-tuning/ y_val.pkl
/kaggle/input/notebooks/jwang2025learning/lr-tuning/custom.css
/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/i

In [3]:
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
import numpy as np

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [5]:
df_test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
test_ids = df_test["id"]                # ← save before basic_cleaning drops it



In [6]:
X_train = np.load("/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_train.npy")
y_train = pd.read_pickle("/kaggle/input/notebooks/jwang2025learning/lr-tuning/y_train.pkl")
X_test = np.load("/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_test.npy")

In [7]:

X_tr = np.load("/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_tr.npy")
y_tr = pd.read_pickle("/kaggle/input/notebooks/jwang2025learning/lr-tuning/y_tr.pkl")
X_val = np.load("/kaggle/input/notebooks/jwang2025learning/lr-tuning/X_val.npy")
y_val = pd.read_pickle("/kaggle/input/notebooks/jwang2025learning/lr-tuning/ y_val.pkl")

In [8]:
!pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 4.7 MB/s eta 0:00:00


In [9]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import numpy as np
from optuna.integration import XGBoostPruningCallback

# ─────────────────────────────────────────────
# 1. DEFINE OBJECTIVE FUNCTION
# ─────────────────────────────────────────────
from optuna.integration import XGBoostPruningCallback
import xgboost as xgb

def objective(trial):

    # -----------------------------
    # 1. Hyperparameter search space
    # -----------------------------
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "predictor": "cpu_predictor",
        "random_state": 42,

        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
    }

  

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_val, label=y_val)

    # -----------------------------
    # 3. Train with early stopping + pruning
    # -----------------------------
    pruning_cb = XGBoostPruningCallback(trial, "validation-auc")

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dvalid, "validation")],
        early_stopping_rounds=50,
        verbose_eval=False,
        callbacks=[pruning_cb],
    )

    # -----------------------------
    # 4. Predict and evaluate
    # -----------------------------
    y_prob = model.predict(dvalid)
    auc = roc_auc_score(y_val, y_prob)

    return auc


# ─────────────────────────────────────────────
# 2. RUN OPTUNA STUDY
# ─────────────────────────────────────────────
study = optuna.create_study(
    direction="maximize",                          # maximize AUC
    sampler=optuna.samplers.TPESampler(seed=42),   # Bayesian optimization
    pruner=optuna.pruners.MedianPruner(),          # stop bad trials early
)

study.optimize(
    objective,
    n_trials=100,          # increase for better results (e.g. 200)
    timeout=600,           # stop after 10 min regardless of n_trials
    show_progress_bar=True,
)

# ─────────────────────────────────────────────
# 3. RESULTS
# ─────────────────────────────────────────────
print(f"\nBest AUC : {study.best_value:.4f}")
print(f"Best Params: {study.best_params}")


# ─────────────────────────────────────────────
# 4. RETRAIN ON FULL TRAINING DATA WITH BEST PARAMS
# ─────────────────────────────────────────────


[I 2026-03-31 21:03:33,036] A new study created in memory with name: no-name-b93152b3-e7cf-42f6-9efe-09f1f5e8afea


  0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:03:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-03-31 21:04:52,906] Trial 0 finished with value: 0.9156284798479999 and parameters: {'learning_rate': 0.008468008575248327, 'max_depth': 10, 'min_child_weight': 8, 'gamma': 2.993292420985183, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'reg_alpha': 0.00019517224641449495, 'reg_lambda': 2.1423021757741068, 'scale_pos_weight': 6.41003510568888}. Best is trial 0 with value: 0.9156284798479999.


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:04:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


...

[I 2026-03-31 21:12:53,849] Trial 45 pruned. Trial was pruned at iteration 3.


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:12:54] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-03-31 21:12:54,754] Trial 46 pruned. Trial was pruned at iteration 2.


/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [21:12:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "predictor" } are not used.

  self.starting_round = model.num_boosted_rounds()


[I 2026-03-31 21:13:38,880] Trial 47 pruned. Trial was pruned at iteration 745.

Best AUC : 0.9160
Best Params: {'learning_rate': 0.01174843954800703, 'max_depth': 5, 'min_child_weight': 7, 'gamma': 0.6974693032602092, 'subsample': 0.6460723242676091, 'colsample_bytree': 0.6831809216468459, 'reg_alpha': 0.019069966103000432, 'reg_lambda': 0.8431013932082461, 'scale_pos_weight': 2.7970640394252375}


In [10]:

best_xgb = XGBClassifier(**study.best_params, eval_metric="auc", random_state=42)
best_xgb.fit(X_train, y_train)
# Evaluate on hold-out test set
y_prob_tuned = best_xgb.predict_proba(X_val)[:, 1]
np.save("xgb_val_predict.npy",y_prob_tuned)
print(f"\nTuned XGBoost AUC on test set: {roc_auc_score(y_val, y_prob_tuned):.4f}")


# ─────────────────────────────────────────────
# 5. GENERATE SUBMISSION
# ─────────────────────────────────────────────
y_prob_final = best_xgb.predict_proba(X_test)[:, 1]

np.save("xgb_best_predict.npy", y_prob_final)

submission = pd.DataFrame({
    "id":    test_ids,
    "Churn": y_prob_final
})

submission.to_csv("submission.csv", index=False)
print(submission["Churn"].value_counts())


Tuned XGBoost AUC on test set: 0.9097
Churn
0.141625    16006
0.144834     4746
0.142700     3817
0.160747     3487
0.160579     3478
            ...  
0.559523        1
0.448912        1
0.351021        1
0.614508        1
0.489101        1
Name: count, Length: 46960, dtype: int64
